# Graph Representation Learning on a Heterogeneous Urban Knowledge Graph
### A step-by-step, theory-annotated benchmark on the pykci CityGML + OSM graph (Hamburg HafenCity)

This notebook is a companion to the paper draft. It is meant to be **read alongside the theory**:
each method is preceded by a short markdown explanation, then a code cell you can run and tweak.

**It runs out-of-the-box on a synthetic HafenCity-like graph** (no database needed), and switches to
your real Neo4j graph by flipping one flag and filling in a loader. Everything is written so you can
point it at another city later.

**What it does**
1. Builds a heterogeneous graph (`Building`, `RoofSurface`, `WallSurface`, `GroundSurface`, `GridCell`).
2. Defines the task: **roof-material node classification** on `Building` nodes.
3. Creates two splits: a **random** split and a **spatial-block** split (the honest one).
4. Benchmarks four families: attribute-only MLP; shallow embeddings (node2vec, metapath2vec);
   heterogeneous GNNs (GraphSAGE/GAT via `to_hetero`); a heterogeneous transformer (HGT).
5. Produces the paper's figures (`fig_results.pdf`, `fig_ablation.pdf`, `fig_embed_tsne.pdf`,
   `fig_tradeoff.pdf`) and `results.csv`.

**Install (run once, then restart the kernel).** PyG must match your torch build; node2vec/metapath2vec
need the random-walk kernel from `torch_cluster`.


In [ ]:
# --- install (uncomment on a fresh environment; then restart kernel) ---
# %pip install torch --index-url https://download.pytorch.org/whl/cpu
# %pip install torch_geometric
# # node2vec / metapath2vec random walks:  (match the torch version in the URL)
# %pip install torch_cluster -f https://data.pyg.org/whl/torch-2.3.0+cpu.html
# %pip install scikit-learn matplotlib pandas numpy

import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

import torch_geometric
from torch_geometric.data import HeteroData

print("torch", torch.__version__, "| PyG", torch_geometric.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. The one idea behind all node embeddings

Every method below fits the **encoder–decoder** view (Hamilton, Ying & Leskovec 2017): an *encoder*
maps each node to a vector, trained so that vector geometry reflects the graph. Methods differ on two axes.

**Axis 1 — encoder type.**
- *Shallow / lookup* (DeepWalk, node2vec, metapath2vec): a free embedding vector **per node**, learned
  directly. Uses **no node features**. **Transductive** — cannot embed a node it never saw.
- *Message-passing GNN* (GraphSAGE, GAT, R-GCN, HGT): the embedding is a **function of a node's features
  and its neighbourhood**. Uses features. **Inductive** — can embed unseen nodes (e.g. a new building).

**Axis 2 — training signal.**
- *Unsupervised / structure-only*: reconstruct proximity or random-walk co-occurrence (node2vec).
- *Supervised*: predict a label end-to-end (the GNNs here).
- *Self-supervised*: contrastive / masked reconstruction (DGI, GraphMAE — the follow-up paper).

A benchmark's value is spanning these axes, so you can see **what each ingredient buys**:
features (attribute-only MLP), topology (node2vec/metapath2vec), and type-awareness (heterogeneous GNNs).


In [ ]:
# ------------------------- experiment configuration -------------------------
CITY          = "HafenCity"     # label only; swap the loader below for real data
USE_SYNTHETIC = True            # True -> runs offline on a generated graph
                                # False -> pull your real graph from Neo4j (template below)

# Neo4j (only used if USE_SYNTHETIC=False)
import os
NEO4J_URI  = os.environ.get("NEO4J_URI",  "bolt://localhost:7687")
NEO4J_USER = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PWD  = os.environ.get("NEO4J_PASSWORD")

# model / training hyperparameters (small graph -> fast; increase for real data)
EMB_DIM, HIDDEN, GNN_LAYERS, HEADS, DROPOUT = 64, 64, 2, 4, 0.3
LR, GNN_EPOCHS, N2V_EPOCHS = 0.01, 150, 5
SEEDS = [0, 1, 2, 3, 4]

# random-walk params (node2vec / metapath2vec)
WALK_LENGTH, CONTEXT_SIZE, WALKS_PER_NODE, P, Q = 20, 7, 10, 1.0, 1.0
N_CLASSES = None  # set from data after loading

## 2. Build the heterogeneous graph

We mirror the `pykci` schema: `Building` nodes carry the prediction targets and geometric features;
they decompose into CityGML boundary surfaces (`RoofSurface`, `WallSurface`, `GroundSurface`), sit in a
`GridCell`, and connect to spatially adjacent buildings.

The synthetic generator plants **spatial autocorrelation** in the roof-material labels on purpose:
materials depend on the building's quadrant plus noise. That is what makes the random-vs-spatial split
gap (Section 3) visible — exactly the effect you must control for with real city data.


In [ ]:
from sklearn.neighbors import NearestNeighbors

def make_synthetic_city(n_buildings=800, grid=20, walls_per_building=4, k_adj=6, seed=0):
    rng = np.random.default_rng(seed)

    # buildings on a 2D grid; coordinates drive both structure and labels
    coords  = rng.uniform(0, grid, size=(n_buildings, 2))
    height  = rng.gamma(2.0, 4.0, n_buildings) + 3.0
    area    = rng.gamma(2.0, 60.0, n_buildings) + 40.0
    orient  = rng.uniform(0, 180, n_buildings)
    storeys = np.clip(np.round(height / 3.0), 1, None)

    # roof material WITH spatial autocorrelation (quadrant prior + noise)
    qx = (coords[:, 0] > grid / 2).astype(int)
    qy = (coords[:, 1] > grid / 2).astype(int)
    region  = qx * 2 + qy                          # 0..3
    logits  = np.eye(4)[region] * 2.5 + rng.normal(0, 1.0, size=(n_buildings, 4))
    material = logits.argmax(1).astype(int)

    def z(a):
        a = np.asarray(a, dtype=np.float32)
        return (a - a.mean(0)) / (a.std(0) + 1e-6)

    d = HeteroData()
    d["Building"].x   = torch.tensor(z(np.stack([height, area, orient, storeys], 1)), dtype=torch.float)
    d["Building"].y   = torch.tensor(material, dtype=torch.long)
    d["Building"].pos = torch.tensor(coords, dtype=torch.float)   # used only for the spatial split

    # boundary surfaces: 1 roof + 1 ground + W walls per building
    d["RoofSurface"].x   = torch.tensor(z(np.stack([area*rng.uniform(0.8,1.2,n_buildings),
                                                    rng.uniform(0,45,n_buildings)], 1)), dtype=torch.float)
    d["GroundSurface"].x = torch.tensor(z(area.reshape(-1,1)), dtype=torch.float)
    n_walls = n_buildings*walls_per_building
    d["WallSurface"].x   = torch.tensor(z(np.stack([rng.gamma(2.0,20.0,n_walls)+10.0,
                                                    rng.uniform(0,360,n_walls)], 1)), dtype=torch.float)

    # grid cells (regular tessellation)
    cx, cy = np.meshgrid(np.arange(grid)+0.5, np.arange(grid)+0.5)
    cell_xy = np.stack([cx.ravel(), cy.ravel()], 1)
    d["GridCell"].x = torch.tensor(z(cell_xy), dtype=torch.float)

    b = np.arange(n_buildings)
    E = lambda s, t: torch.tensor(np.stack([s, t], 0), dtype=torch.long)

    d["Building","hasRoof","RoofSurface"].edge_index    = E(b, b)          # roof i belongs to building i
    d["Building","hasGround","GroundSurface"].edge_index = E(b, b)
    d["Building","hasWall","WallSurface"].edge_index     = E(np.repeat(b, walls_per_building),
                                                             np.arange(n_walls))
    cell_idx = (np.clip(coords[:,0].astype(int),0,grid-1)*grid
                + np.clip(coords[:,1].astype(int),0,grid-1))
    d["Building","inCell","GridCell"].edge_index = E(b, cell_idx)

    nn = NearestNeighbors(n_neighbors=k_adj+1).fit(coords)
    _, idx = nn.kneighbors(coords)
    d["Building","adjacent","Building"].edge_index = E(np.repeat(b, k_adj), idx[:,1:].reshape(-1))

    # add reverse relations so messages can flow both ways / metapaths can return
    for (s, r, t) in list(d.edge_types):
        if r == "adjacent":
            continue
        d[t, "rev_"+r, s].edge_index = d[s, r, t].edge_index.flip(0)
    ei = d["Building","adjacent","Building"].edge_index          # make adjacency symmetric
    d["Building","adjacent","Building"].edge_index = torch.cat([ei, ei.flip(0)], 1)
    return d

### 2b. Real data: Neo4j loader template

When you are ready, set `USE_SYNTHETIC = False` and fill this in for your schema. The goal is simply to
return a `HeteroData` **shaped like the synthetic one** (same node/edge type names), so every method below
works unchanged. To reuse the spatial split, attach `data['Building'].pos` (x, y coordinates).


In [ ]:
def load_from_neo4j(uri=NEO4J_URI, user=NEO4J_USER, pwd=NEO4J_PWD):
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver(uri, auth=(user, pwd))
    d = HeteroData()
    with driver.session() as s:
        # ---- Buildings: id, numeric features, label, coordinates ----
        rows = s.run(
            "MATCH (b:Building) "
            "RETURN id(b) AS id, b.measuredHeight AS height, b.footprintArea AS area, "
            "       b.orientation AS orient, b.storeys AS storeys, "
            "       b.roofMaterial AS material, b.x AS x, b.y AS y"
        ).data()
        # df = pd.DataFrame(rows)
        # 1) map id -> contiguous index; 2) build feature matrix (z-score numerics,
        #    one-hot categoricals); 3) encode material -> int label;
        # 4) fill d['Building'].x / .y / .pos
        #
        # Repeat for RoofSurface / WallSurface / GroundSurface / GridCell.
        # Edges, e.g.:
        #   MATCH (b:Building)-[:hasRoofSurface]->(r:RoofSurface) RETURN id(b), id(r)
        # then remap endpoint ids to the per-type contiguous indices and set
        #   d['Building','hasRoof','RoofSurface'].edge_index = ...
        # Finally add reverse relations exactly as in make_synthetic_city().
        pass
    driver.close()
    raise NotImplementedError("Fill in the Cypher->HeteroData mapping, then delete this raise.")
    return d

In [ ]:
# ------------------------- build + inspect the graph -------------------------
data = make_synthetic_city(n_buildings=800, grid=20, seed=0) if USE_SYNTHETIC else load_from_neo4j()
N_CLASSES = int(data["Building"].y.max().item()) + 1
print(data)

# ---- dataset statistics (maps to paper Table 1) ----
print("\nNode types:")
for nt in data.node_types:
    print(f"  {nt:14s} feat_dim={data[nt].x.size(1):>2d}  count={data[nt].num_nodes}")
print("Edge types:")
for (s, r, t) in data.edge_types:
    print(f"  ({s} -{r}-> {t})".ljust(52), "count=", data[s, r, t].edge_index.size(1))

y = data["Building"].y.numpy()
u, c = np.unique(y, return_counts=True)
print("\nclasses:", N_CLASSES, "| distribution:", dict(zip(u.tolist(), c.tolist())),
      "| majority frac: %.3f" % (c.max()/c.sum()))

## 3. Splits: random vs. spatial (the honest one)

Nearby buildings share materials, heights, and uses — strong **spatial autocorrelation**. A random node
split then leaks test labels through the graph: a train neighbour effectively reveals its test
neighbour's label, and every method looks great for the wrong reason.

We therefore also build a **spatial-block split**: we tile space and assign *whole tiles* to
train/val/test, so regions are geographically disjoint. The gap between the two is the deployment-relevant
number — can the model label a **previously unseen part of the city**? Expect scores to drop under the
spatial split; that drop is a result, not a bug.


In [ ]:
def make_random_split(n, seed=0, frac=(0.6, 0.2, 0.2)):
    rng = np.random.default_rng(seed); idx = rng.permutation(n)
    a, b = int(frac[0]*n), int((frac[0]+frac[1])*n)
    def mask(ix):
        m = torch.zeros(n, dtype=torch.bool); m[ix] = True; return m
    return mask(idx[:a]), mask(idx[a:b]), mask(idx[b:])

def make_spatial_split(pos, seed=0, frac=(0.6, 0.2, 0.2), blocks=8):
    rng = np.random.default_rng(seed); p = pos.numpy()
    xs = np.linspace(p[:,0].min(), p[:,0].max()+1e-6, blocks+1)
    ys = np.linspace(p[:,1].min(), p[:,1].max()+1e-6, blocks+1)
    tile = (np.clip(np.digitize(p[:,0], xs)-1, 0, blocks-1)*blocks
            + np.clip(np.digitize(p[:,1], ys)-1, 0, blocks-1))
    tiles = rng.permutation(np.unique(tile))
    a, b = int(frac[0]*len(tiles)), int((frac[0]+frac[1])*len(tiles))
    sets = [set(tiles[:a]), set(tiles[a:b]), set(tiles[b:])]
    return tuple(torch.tensor(np.isin(tile, list(S))) for S in sets)

n_b = data["Building"].num_nodes
splits = {
    "random":  make_random_split(n_b, seed=0),
    "spatial": make_spatial_split(data["Building"].pos, seed=0, blocks=8),
}
for name, (tr, va, te) in splits.items():
    print(f"{name:8s} train/val/test = {int(tr.sum())}/{int(va.sum())}/{int(te.sum())}")

## 4. Evaluation harness

Fixed splits, shared across methods; report **mean ± std over seeds**. Roof material is imbalanced, so we
track **macro-F1** (equal weight per class) alongside accuracy. Shallow embeddings are evaluated with a
frozen-embedding **logistic-regression probe**; GNNs are trained end-to-end. We also time each method for
the accuracy-vs-cost figure.


In [ ]:
def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def scores(y_true, y_pred):
    return {"acc": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro"),
            "weight_f1": f1_score(y_true, y_pred, average="weighted")}

def agg_seeds(run_fn, split):
    t0 = time.time()
    per = [run_fn(split, s) for s in SEEDS]
    secs = (time.time() - t0) / len(SEEDS)
    out = {}
    for k in per[0]:
        v = [p[k] for p in per]
        out[k], out[k + "_std"] = float(np.mean(v)), float(np.std(v))
    out["seconds"] = secs
    return out

RESULTS = []
def record(method, split, agg):
    RESULTS.append({"method": method, "split": split, **agg})
    print(f"  {method:20s} [{split:7s}] acc={agg['acc']:.3f}  macroF1={agg['macro_f1']:.3f}"
          f"  ({agg['seconds']:.1f}s/seed)")

def probe(emb, split):
    y = data["Building"].y.numpy()
    tr, va, te = [m.numpy() for m in splits[split]]
    clf = LogisticRegression(max_iter=1000).fit(emb[tr], y[tr])
    return scores(y[te], clf.predict(emb[te]))

## 5. Family F0 — attribute-only MLP (no graph)

The most important baseline. If a plain MLP on node attributes already does well, the graph adds little;
if it lags the GNNs, the graph is carrying real signal. It also sets the "features only" bar for the
ablation in Section 11.


In [ ]:
class MLP(torch.nn.Module):
    def __init__(self, d_in, d_hid, d_out):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(d_in, d_hid), torch.nn.ReLU(), torch.nn.Dropout(DROPOUT),
            torch.nn.Linear(d_hid, d_out))
    def forward(self, x): return self.net(x)

def run_mlp(split, seed):
    set_seed(seed)
    x = data["Building"].x.to(device); y = data["Building"].y.to(device)
    tr, va, te = (m.to(device) for m in splits[split])
    model = MLP(x.size(1), HIDDEN, N_CLASSES).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=5e-4)
    for _ in range(GNN_EPOCHS):
        model.train(); opt.zero_grad()
        F.cross_entropy(model(x)[tr], y[tr]).backward(); opt.step()
    model.eval()
    with torch.no_grad(): pred = model(x).argmax(1)
    return scores(y[te].cpu().numpy(), pred[te].cpu().numpy())

print("F0: MLP (attributes only)")
for split in ["random", "spatial"]:
    record("MLP (attr-only)", split, agg_seeds(run_mlp, split))

## 6. Family F1a — node2vec (shallow, structure-only)

**The concept you want to internalise.** node2vec (Grover & Leskovec 2016) runs biased random walks,
treats each walk like a sentence, and applies **skip-gram**: nodes that co-occur on walks get similar
vectors. The encoder is just a **lookup table** (one vector per node); it uses **no features** and is
**transductive**. `p` and `q` bias walks toward breadth (homophily/communities) or depth (structural
roles). We collapse node types into one graph, train, then classify `Building` embeddings with a probe.

Requires `torch_cluster` for the random-walk kernel.


In [ ]:
from torch_geometric.nn import Node2Vec

def homogeneous_edge_index(data):
    offset, cur = {}, 0
    for nt in data.node_types:
        offset[nt] = cur; cur += data[nt].num_nodes
    eis = []
    for (s, r, t) in data.edge_types:
        ei = data[s, r, t].edge_index.clone()
        ei[0] += offset[s]; ei[1] += offset[t]; eis.append(ei)
    return torch.cat(eis, 1), offset, cur

hom_ei, OFFSET, TOTAL_NODES = homogeneous_edge_index(data)

def run_node2vec(split, seed):
    set_seed(seed)
    m = Node2Vec(hom_ei, embedding_dim=EMB_DIM, walk_length=WALK_LENGTH,
                 context_size=CONTEXT_SIZE, walks_per_node=WALKS_PER_NODE,
                 num_negative_samples=1, p=P, q=Q, num_nodes=TOTAL_NODES,
                 sparse=True).to(device)
    loader = m.loader(batch_size=128, shuffle=True, num_workers=0)
    opt = torch.optim.SparseAdam(list(m.parameters()), lr=0.01)
    m.train()
    for _ in range(N2V_EPOCHS):
        for pos_rw, neg_rw in loader:
            opt.zero_grad(); m.loss(pos_rw.to(device), neg_rw.to(device)).backward(); opt.step()
    m.eval()
    with torch.no_grad(): z = m().cpu().numpy()
    b0, nb = OFFSET["Building"], data["Building"].num_nodes
    return probe(z[b0:b0+nb], split)

print("F1: node2vec (structure-only) + logistic-regression probe")
for split in ["random", "spatial"]:
    record("node2vec", split, agg_seeds(run_node2vec, split))

## 7. Family F1b — metapath2vec (shallow, heterogeneous)

metapath2vec (Dong et al. 2017) is node2vec made type-aware: walks follow a **metapath**, a template of
relation types, so co-occurrence respects the schema. Here we use
`Building -hasRoof-> Roof -rev_hasRoof-> Building -inCell-> Cell -rev_inCell-> Building`, i.e. buildings
are "similar" if they share roof-surface structure and grid-cell co-location. Compare it to node2vec to
see whether **typed** topology beats untyped.


In [ ]:
from torch_geometric.nn import MetaPath2Vec

metapath = [
    ("Building", "hasRoof", "RoofSurface"),
    ("RoofSurface", "rev_hasRoof", "Building"),
    ("Building", "inCell", "GridCell"),
    ("GridCell", "rev_inCell", "Building"),
]

def run_metapath2vec(split, seed):
    set_seed(seed)
    m = MetaPath2Vec(data.edge_index_dict, embedding_dim=EMB_DIM, metapath=metapath,
                     walk_length=WALK_LENGTH, context_size=CONTEXT_SIZE,
                     walks_per_node=WALKS_PER_NODE, num_negative_samples=1,
                     num_nodes_dict={nt: data[nt].num_nodes for nt in data.node_types},
                     sparse=True).to(device)
    loader = m.loader(batch_size=128, shuffle=True, num_workers=0)
    opt = torch.optim.SparseAdam(list(m.parameters()), lr=0.01)
    m.train()
    for _ in range(N2V_EPOCHS):
        for pos_rw, neg_rw in loader:
            opt.zero_grad(); m.loss(pos_rw.to(device), neg_rw.to(device)).backward(); opt.step()
    m.eval()
    with torch.no_grad(): emb = m("Building").cpu().numpy()
    return probe(emb, split)

print("F1: metapath2vec (heterogeneous structure-only) + probe")
for split in ["random", "spatial"]:
    record("metapath2vec", split, agg_seeds(run_metapath2vec, split))

## 8. Family F2 — heterogeneous GNNs via `to_hetero` (GraphSAGE / GAT)

Now features enter. We write a plain 2-layer GNN and let PyG's `to_hetero` duplicate its message passing
**per relation type** (a separate transform for `hasRoof`, `rev_inCell`, `adjacent`, ...), summing the
messages arriving at each node. This is inductive and uses attributes. GraphSAGE aggregates neighbour
features; GAT weights neighbours by learned attention. Per Lv et al. (2021), a properly tuned model of
this kind is a strong baseline that fancier heterogeneous models must beat.


In [ ]:
from torch_geometric.nn import SAGEConv, GATConv, Linear, to_hetero

class GNN(torch.nn.Module):
    def __init__(self, hidden, out, conv="sage"):
        super().__init__()
        if conv == "gat":
            mk = lambda: GATConv((-1, -1), hidden, heads=HEADS, concat=False, add_self_loops=False)
        else:
            mk = lambda: SAGEConv((-1, -1), hidden)
        self.conv1, self.conv2, self.lin = mk(), mk(), Linear(hidden, out)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=DROPOUT, training=self.training)
        x = self.conv2(x, edge_index).relu()
        return self.lin(x)

def _device_graph():
    x_dict  = {k: v.to(device) for k, v in data.x_dict.items()}
    ei_dict = {k: v.to(device) for k, v in data.edge_index_dict.items()}
    return x_dict, ei_dict

def run_hetero_gnn(split, seed, conv="sage"):
    set_seed(seed)
    x_dict, ei_dict = _device_graph()
    y = data["Building"].y.to(device)
    tr, va, te = (m.to(device) for m in splits[split])
    model = to_hetero(GNN(HIDDEN, N_CLASSES, conv), data.metadata(), aggr="sum").to(device)
    with torch.no_grad(): model(x_dict, ei_dict)          # initialise lazy params
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=5e-4)
    for _ in range(GNN_EPOCHS):
        model.train(); opt.zero_grad()
        out = model(x_dict, ei_dict)["Building"]
        F.cross_entropy(out[tr], y[tr]).backward(); opt.step()
    model.eval()
    with torch.no_grad(): pred = model(x_dict, ei_dict)["Building"].argmax(1)
    return scores(y[te].cpu().numpy(), pred[te].cpu().numpy())

print("F2: heterogeneous GraphSAGE (via to_hetero)")
for split in ["random", "spatial"]:
    record("GraphSAGE (hetero)", split, agg_seeds(lambda sp, se: run_hetero_gnn(sp, se, "sage"), split))

print("F2: heterogeneous GAT (via to_hetero)")
for split in ["random", "spatial"]:
    record("GAT (hetero)", split, agg_seeds(lambda sp, se: run_hetero_gnn(sp, se, "gat"), split))

## 9. Family F3 — Heterogeneous Graph Transformer (HGT)

HGT (Hu et al. 2020) uses **type-aware transformer attention**: separate query/key/value projections per
node and edge type, so the model learns how much each relation matters without hand-designed metapaths.
It is the most expressive model here. We project each node type to a common width, stack HGT layers, then
read out the `Building` nodes.


In [ ]:
from torch_geometric.nn import HGTConv, Linear as PygLinear

class HGT(torch.nn.Module):
    def __init__(self, hidden, out, heads, layers, metadata):
        super().__init__()
        self.lin_in = torch.nn.ModuleDict({nt: PygLinear(-1, hidden) for nt in metadata[0]})
        self.convs = torch.nn.ModuleList([HGTConv(hidden, hidden, metadata, heads)
                                          for _ in range(layers)])
        self.lin_out = PygLinear(hidden, out)
    def encode(self, x_dict, ei_dict):
        h = {nt: self.lin_in[nt](x).relu() for nt, x in x_dict.items()}
        for conv in self.convs:
            h = conv(h, ei_dict)
        return h["Building"]
    def forward(self, x_dict, ei_dict):
        return self.lin_out(self.encode(x_dict, ei_dict))

def run_hgt(split, seed):
    set_seed(seed)
    x_dict, ei_dict = _device_graph()
    y = data["Building"].y.to(device)
    tr, va, te = (m.to(device) for m in splits[split])
    model = HGT(HIDDEN, N_CLASSES, HEADS, GNN_LAYERS, data.metadata()).to(device)
    with torch.no_grad(): model(x_dict, ei_dict)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=5e-4)
    for _ in range(GNN_EPOCHS):
        model.train(); opt.zero_grad()
        F.cross_entropy(model(x_dict, ei_dict)[tr], y[tr]).backward(); opt.step()
    model.eval()
    with torch.no_grad(): pred = model(x_dict, ei_dict).argmax(1)
    return scores(y[te].cpu().numpy(), pred[te].cpu().numpy())

print("F3: Heterogeneous Graph Transformer (HGT)")
for split in ["random", "spatial"]:
    record("HGT", split, agg_seeds(run_hgt, split))

## 10. Extend the suite: R-GCN, HAN, Simple-HGN (scaffolds)

To complete the paper's method table, add these. They plug into the same harness (define
`run_<x>(split, seed)` returning `scores(...)`, then `record(...)`).

- **R-GCN** — `torch_geometric.nn.RGCNConv`; per-relation weight matrices. Easiest via
  `data.to_homogeneous()` (gives `edge_type`) after projecting each node type to a shared feature width.
- **HAN** — `torch_geometric.nn.HANConv`; needs a list of metapaths (reuse the one from Section 7 plus a
  `Building -adjacent-> Building` path). Metapath-level attention.
- **Simple-HGN** — GAT + learnable edge-type embedding + residual + L2-normalised output. The strong
  baseline from Lv et al. (2021); reimplement on top of the GAT variant in Section 8.


In [ ]:
# --- scaffold only; uncomment and complete, then add to RESULTS ---
# from torch_geometric.nn import HANConv, RGCNConv
#
# def run_han(split, seed):
#     metapaths = [metapath, [("Building","adjacent","Building")]]
#     ... # build HANConv model over metapaths, train end-to-end, return scores(...)
#
# for split in ["random", "spatial"]:
#     record("HAN", split, agg_seeds(run_han, split))
print("scaffolds ready — implement to fill the R-GCN / HAN / Simple-HGN rows of Table 2")

## 11. Results and figures

Assemble `results.csv` (paper Tables 2–3) and export the four figures. The figures are written to the
working directory as PDFs with the exact names referenced by the paper draft.


In [ ]:
df = pd.DataFrame(RESULTS)
df.to_csv("results.csv", index=False)
print(df[["method", "split", "acc", "macro_f1", "seconds"]].to_string(index=False))

In [ ]:
# ---- Figure 3: macro-F1 by method, random vs spatial (paper fig_results.pdf) ----
methods = df[df.split == "random"]["method"].tolist()
x = np.arange(len(methods)); w = 0.38
fig, ax = plt.subplots(figsize=(9, 4.6))
for i, split in enumerate(["random", "spatial"]):
    sub = df[df.split == split].set_index("method").reindex(methods)
    ax.bar(x + (i - 0.5)*w, sub["macro_f1"], width=w,
           yerr=sub.get("macro_f1_std"), capsize=3, label=split)
ax.set_xticks(x); ax.set_xticklabels(methods, rotation=30, ha="right")
ax.set_ylabel("macro-F1"); ax.set_ylim(0, 1)
ax.set_title(f"Roof-material classification — {CITY}"); ax.legend()
fig.tight_layout(); fig.savefig("fig_results.pdf"); plt.show()

In [ ]:
# ---- Figure 4: ablation features vs structure vs both (paper fig_ablation.pdf) ----
gnn_methods = ["GraphSAGE (hetero)", "GAT (hetero)", "HGT"]
r = df[df.split == "random"].set_index("method")
abl = {
    "attributes only\n(MLP)":          r.loc["MLP (attr-only)", "macro_f1"],
    "structure only\n(metapath2vec)":  r.loc["metapath2vec", "macro_f1"],
    "attributes + structure\n(best GNN)": r.reindex(gnn_methods)["macro_f1"].max(),
}
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(list(abl.keys()), list(abl.values()), color=["#bbbbbb", "#8888bb", "#33aa66"])
ax.set_ylabel("macro-F1 (random split)"); ax.set_ylim(0, 1)
ax.set_title("What carries the signal?")
fig.tight_layout(); fig.savefig("fig_ablation.pdf"); plt.show()

In [ ]:
# ---- Figure 5: t-SNE of embeddings coloured by roof material (fig_embed_tsne.pdf) ----
from sklearn.manifold import TSNE

# structure-only embedding (node2vec)
set_seed(0)
m = Node2Vec(hom_ei, embedding_dim=EMB_DIM, walk_length=WALK_LENGTH, context_size=CONTEXT_SIZE,
             walks_per_node=WALKS_PER_NODE, num_negative_samples=1, p=P, q=Q,
             num_nodes=TOTAL_NODES, sparse=True).to(device)
ld = m.loader(batch_size=128, shuffle=True, num_workers=0)
op = torch.optim.SparseAdam(list(m.parameters()), lr=0.01)
for _ in range(N2V_EPOCHS):
    for pr, nr in ld:
        op.zero_grad(); m.loss(pr.to(device), nr.to(device)).backward(); op.step()
b0, nb = OFFSET["Building"], data["Building"].num_nodes
emb_n2v = m().detach().cpu().numpy()[b0:b0+nb]

# features+structure embedding (trained HGT, penultimate)
set_seed(0); x_dict, ei_dict = _device_graph(); yb = data["Building"].y.to(device)
tr = splits["random"][0].to(device)
hgt = HGT(HIDDEN, N_CLASSES, HEADS, GNN_LAYERS, data.metadata()).to(device)
with torch.no_grad(): hgt(x_dict, ei_dict)
op = torch.optim.Adam(hgt.parameters(), lr=LR, weight_decay=5e-4)
for _ in range(GNN_EPOCHS):
    hgt.train(); op.zero_grad()
    F.cross_entropy(hgt(x_dict, ei_dict)[tr], yb[tr]).backward(); op.step()
hgt.eval()
with torch.no_grad(): emb_hgt = hgt.encode(x_dict, ei_dict).cpu().numpy()

ylab = data["Building"].y.numpy()
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, emb, name in zip(axes, [emb_n2v, emb_hgt],
                         ["node2vec (structure only)", "HGT (features + structure)"]):
    z2 = TSNE(n_components=2, init="pca", perplexity=30, random_state=0).fit_transform(emb)
    sc = ax.scatter(z2[:, 0], z2[:, 1], c=ylab, cmap="tab10", s=10)
    ax.set_title(name); ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(*sc.legend_elements(), title="material", fontsize=8, loc="best")
fig.suptitle(f"Building embeddings coloured by roof material — {CITY}")
fig.tight_layout(); fig.savefig("fig_embed_tsne.pdf"); plt.show()

In [ ]:
# ---- Figure 6 (optional): accuracy vs training cost (fig_tradeoff.pdf) ----
sub = df[df.split == "random"]
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(sub["seconds"], sub["macro_f1"])
for _, row in sub.iterrows():
    ax.annotate(row["method"], (row["seconds"], row["macro_f1"]),
                fontsize=8, xytext=(4, 2), textcoords="offset points")
ax.set_xscale("log"); ax.set_xlabel("training time per run (s, log scale)")
ax.set_ylabel("macro-F1 (random split)"); ax.set_title("Accuracy vs. cost")
fig.tight_layout(); fig.savefig("fig_tradeoff.pdf"); plt.show()

## 12. Adapting to another city (or your real HafenCity graph)

1. Set `USE_SYNTHETIC = False` and complete `load_from_neo4j()` so it returns a `HeteroData` with the same
   node/edge type names. Attach `data['Building'].pos` if you want the spatial split.
2. **Featurise honestly.** Z-score numeric attributes; one-hot categoricals. **Never** feed a feature
   derived from the target (e.g. a material-derived indicator when predicting material).
3. Set `N_CLASSES` from the data (already done automatically after loading).
4. Increase `GNN_EPOCHS`, `N2V_EPOCHS`, `EMB_DIM`, and add early stopping on the validation mask for a
   larger graph.
5. Swap the target: point `data['Building'].y` at building use (classification) or storeys/height
   (switch the loss to `F.mse_loss` and report MAE/RMSE/R^2) to reproduce the secondary tasks.
6. Re-run top to bottom; `results.csv` and the four PDFs regenerate for the new city.

**Reading pairing.** Run Sections 5–6 while reading node2vec; Section 7 with the metapath2vec paper;
Sections 8–9 with the GraphSAGE / HGT papers; and read Lv et al. (2021) before interpreting the
heterogeneous-vs-homogeneous comparison in Section 11.


## 13. Real Neo4j data — S5 roof-type classification (provenance-agnostic)

Everything above ran on the synthetic graph. This section runs the **real** benchmark on the live
Neo4j KG, using the repo's reusable modules — written to be **portable across cities** and **scalable
to the full dataset**:

- `neo4j_loader.load_from_neo4j()` introspects the schema and returns a `HeteroData`; on another city
  it silently **drops absent node types / features / labels** (see `real.provenance_report`), and the
  lazy PyG layers absorb the changed feature widths.
- `splits.make_splits()` gives two protocols (S4): **`random`** (i.i.d. hold-out, full 5-class label —
  *deceptive*: leaks through spatial autocorrelation) and **`spatial_cv`** (spatial K-fold CV on the
  merged 3-class label {Flat / Gable-family / Other} — *honest*: pooled out-of-fold macro-F1, so every
  building is tested once). Majority-class floor ≈ **macro-F1 0.286**.
- `s5_train.run_method()` trains the attribute probe (F0) and heterogeneous GraphSAGE (F2), reporting
  **macro-F1 mean±std** over seeds and **per-class F1** (never accuracy — §1 pt 8).

Set `NEO4J_PASSWORD` in your shell **before** launching Jupyter; don't hard-code it here.

In [ ]:
# --- load the live graph (portable: absent features/types are dropped, not assumed) ---
import os
from neo4j_loader import load_from_neo4j
from s5_train import run_method, print_result, pick_device

os.environ.setdefault("NEO4J_URI",  "neo4j://127.0.0.1:7687")
os.environ.setdefault("NEO4J_USER", "neo4j")
# os.environ["NEO4J_PASSWORD"] must already be set in the environment.

real = load_from_neo4j()          # prints per-type shapes + assertions
print("device:", pick_device())
print("kept/dropped per city:", real.provenance_report["nodes"].keys(),
      "| dropped node types:", real.provenance_report["dropped_node_types"])

In [ ]:
# --- S5: provenance-AGNOSTIC roof-type classification ---
SEEDS_REAL = (0, 1, 2, 3, 4)
res_probe = run_method(real, method="probe", seeds=SEEDS_REAL)
res_sage  = run_method(real, method="gnn",   seeds=SEEDS_REAL, epochs=150)

print("S5 — provenance-AGNOSTIC (random=5-class deceptive; spatial_cv=3-class honest; floor≈0.286)")
print_result("F0 attribute probe", res_probe)
print_result("F2 GraphSAGE (hetero)", res_sage)

### Running on the full dataset (96 GB workstation) and other cities

- **Full DB:** full-batch fits large graphs on 96 GB; if VRAM is tight, use the sampled path:
  `run_method(real, "gnn", full_batch=False, batch_size=4096, num_neighbors=[15, 10], amp=True)`.
  Keep the master `HeteroData` on CPU (the loop builds per-run device tensors — §3 item 9).
- **Another city:** point `NEO4J_*` at it and re-run — no code change. Missing facades / roof material /
  whole node types are handled by the loader; check `real.provenance_report` to see what this city had.
- **S6/S6b (provenance-AWARE) are implemented:** `run_method(real, "gnn_aware")` runs Eq.1's
  confidence-weighted (`jaccard`=w_ce), source-typed (`W_{s(e)}`) aggregation on `x_prov` features.
  The **delta vs agnostic**, sliced on the OSM-vs-CityGML disagreement subset, is the paper's thesis.

### S6b — provenance-AWARE (Eq.1) and the disagreement slice

`gnn_aware` realizes  h_c' = σ(W_self h_c + Σ (w_ce/Σw)·W_{s(e)} h_e): confidence-weighted
(`jaccard`), source-typed aggregation on the coverage/agreement features. Compare it to the agnostic
GNN and, crucially, slice the delta where OSM roof evidence **disagrees** with the canonical CityGML
roof type — where §1 pt 7 predicts the largest gain.

In [ ]:
from s5_train import oof_predictions
from splits import merged_3class
import numpy as np, os
from neo4j import GraphDatabase

res_aware = run_method(real, method="gnn_aware", seeds=SEEDS_REAL, epochs=150)
print("S6b — provenance-AWARE (Eq.1)")
print_result("F3 GNN AWARE", res_aware)
d = res_aware["spatial_cv"]["macro_mean"] - res_sage["spatial_cv"]["macro_mean"]
print(f"  spatial_cv Δ(aware - agnostic) = {d:+.3f}")

# --- disagreement slice (per-node accuracy, pooled OOF, averaged over seeds) ---
y3, _ = merged_3class(real)
drv = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]))
with drv.session() as s:
    rows = s.run("MATCH (b:Building) RETURN elementId(b) AS eid, coalesce(b.osm_roof_shape,'') AS osm "
                 "ORDER BY eid").data()
drv.close()
OSM3 = {"flat":0,"gabled":1,"hipped":1,"pitched":1,"pyramidal":2}
osm = np.array([OSM3.get(r["osm"], -1) for r in rows]); has = osm>=0
disagree, agree = has & (osm!=y3), has & (osm==y3)
n = real["Building"].num_nodes; ca=cw=cnt=np.zeros(n)
ca, cw, cnt = np.zeros(n), np.zeros(n), np.zeros(n)
for seed in SEEDS_REAL:
    yt,oa,_,_ = oof_predictions(real,"gnn",seed,epochs=150)
    _, ob,_,_ = oof_predictions(real,"gnn_aware",seed,epochs=150)
    m=oa>=0; ca[m]+=oa[m]==yt[m]; cw[m]+=ob[m]==yt[m]; cnt[m]+=1
aa=np.divide(ca,cnt,out=np.zeros(n),where=cnt>0); ww=np.divide(cw,cnt,out=np.zeros(n),where=cnt>0)
for tag,msk in [("DISAGREE",disagree),("agree",agree),("no_evidence",~has)]:
    if msk.sum(): print(f"  {tag:12s} n={int(msk.sum()):2d}  agnostic={aa[msk].mean():.3f}  aware={ww[msk].mean():.3f}  Δ={ww[msk].mean()-aa[msk].mean():+.3f}")